In [10]:
%pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import re
import time
from pathlib import Path

from google import genai
from google.genai import types

In [3]:
APP_ROOT = Path("/home/dhruva/pop_gemini_direct_pdf_pipeline")

DOC_NAME = "kannada_test_job"
JOB_ROOT = APP_ROOT / "workdir" / DOC_NAME
PAGES_PDF_ROOT = JOB_ROOT / "pages_pdf"

PROMPT_PATH = APP_ROOT / "prompts" / "gemini_direct_pdf_translation_prompt.txt"

MODEL = "gemini-3.1-pro-preview"
START_PAGE = 16
END_PAGE = 20

OVERWRITE_EXISTING = True
MAX_RETRIES = 5
RETRY_WAIT_SECONDS = 15

GEMINI_API_KEY = "ADD YOUR API KEY HERE"

print("Pages PDF root:", PAGES_PDF_ROOT)
print("Prompt path:", PROMPT_PATH)
print("Pages PDF root exists:", PAGES_PDF_ROOT.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("API key found:", bool(GEMINI_API_KEY))

Pages PDF root: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf
Prompt path: /home/dhruva/pop_gemini_direct_pdf_pipeline/prompts/gemini_direct_pdf_translation_prompt.txt
Pages PDF root exists: True
Prompt exists: True
API key found: True


In [4]:
PROMPT = PROMPT_PATH.read_text(encoding="utf-8").strip()
print(PROMPT)

Translate this PDF page into English while preserving the original structure as faithfully as possible.

Follow these rules strictly:
1. Preserve the full content. Do not omit, summarize, simplify, or paraphrase.
2. Preserve headings, numbered lists, bullet points, tables, line breaks, and section order.
3. Preserve tables strictly in valid Markdown table format whenever the source contains a table.
4. Do not convert tables into paragraphs, bullet points, or free text.
5. Preserve the original number of rows and columns as closely as possible.
6. Preserve chemical names, crop names, pest names, units, doses, formulation codes, percentages, and numbers exactly.
7. Do not add content that is not present in the PDF.
8. Output pure Markdown only.
9. Do not use HTML tags such as <div>, <span>, <table>, or styling.
10. If a table is complex, still represent it as a Markdown table instead of prose.
11. Return only the final translated Markdown.

Translate this PDF page into English now.


In [5]:
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not found in environment.")

client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client ready")

Gemini client ready


In [6]:
tools = [
    types.Tool(
        googleSearch=types.GoogleSearch()
    ),
]

generate_content_config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(
        thinking_level="HIGH",
    ),
    tools=tools,
)

In [7]:
def page_sort_key(page_dir: Path):
    m = re.search(r"page_(\d+)$", page_dir.name)
    return int(m.group(1)) if m else page_dir.name


def clean_translated_output(text: str) -> str:
    text = text.strip()

    # remove accidental code fences if model adds them
    text = re.sub(r"^```[a-zA-Z0-9_-]*\n", "", text)
    text = re.sub(r"\n```$", "", text)

    return text.strip() + "\n"


def translate_page_pdf(pdf_path: Path, prompt: str) -> str:
    pdf_bytes = pdf_path.read_bytes()
    collected_text = []
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            for chunk in client.models.generate_content_stream(
                model=MODEL,
                contents=[
                    types.Content(
                        role="user",
                        parts=[
                            types.Part.from_text(text=prompt),
                            types.Part.from_bytes(
                                data=pdf_bytes,
                                mime_type="application/pdf",
                            ),
                        ],
                    ),
                ],
                config=generate_content_config,
            ):
                if text := chunk.text:
                    collected_text.append(text)

            full_text = "".join(collected_text).strip()
            if not full_text:
                raise RuntimeError("Empty translated output returned by Gemini.")

            return clean_translated_output(full_text)

        except Exception as e:
            last_error = e
            print(f"Attempt {attempt} failed for {pdf_path.name}: {e}")
            collected_text = []
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_WAIT_SECONDS)

    raise RuntimeError(f"Failed after {MAX_RETRIES} attempts for {pdf_path}") from last_error

In [8]:
summary = []

for page_num in range(START_PAGE, END_PAGE + 1):
    page_name = f"page_{page_num}"
    page_dir = PAGES_PDF_ROOT / page_name
    pdf_path = page_dir / f"{page_name}.pdf"
    translated_md_path = page_dir / "translated_en.md"
    error_path = page_dir / "translated_en_error.txt"

    if not pdf_path.exists():
        msg = f"PDF not found: {pdf_path}"
        print(msg)
        summary.append({
            "page": page_name,
            "status": "failed",
            "error": msg,
        })
        if page_dir.exists():
            error_path.write_text(msg, encoding="utf-8")
        continue

    if translated_md_path.exists() and not OVERWRITE_EXISTING:
        print(f"Skipping {page_name}: translated_en.md already exists")
        summary.append({
            "page": page_name,
            "status": "skipped",
            "reason": "translated_en.md already exists",
        })
        continue

    print(f"Processing {page_name} ...")

    try:
        translated_text = translate_page_pdf(pdf_path, PROMPT)
        translated_md_path.write_text(translated_text, encoding="utf-8")

        summary.append({
            "page": page_name,
            "status": "success",
            "output_file": str(translated_md_path),
        })
        print(f"Saved: {translated_md_path}")

    except Exception as e:
        err = str(e)
        error_path.write_text(err, encoding="utf-8")
        summary.append({
            "page": page_name,
            "status": "failed",
            "error": err,
            "error_file": str(error_path),
        })
        print(f"Failed: {page_name} -> {err}")

Processing page_16 ...
Saved: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_16/translated_en.md
Processing page_17 ...
Saved: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_17/translated_en.md
Processing page_18 ...
Saved: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_18/translated_en.md
Processing page_19 ...
Saved: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_19/translated_en.md
Processing page_20 ...
Saved: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_20/translated_en.md


In [9]:
summary_path = JOB_ROOT / "step2_direct_pdf_translation_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print(f"Saved summary: {summary_path}")
summary

Saved summary: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/step2_direct_pdf_translation_summary.json


[{'page': 'page_16',
  'status': 'success',
  'output_file': '/home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_16/translated_en.md'},
 {'page': 'page_17',
  'status': 'success',
  'output_file': '/home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_17/translated_en.md'},
 {'page': 'page_18',
  'status': 'success',
  'output_file': '/home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_18/translated_en.md'},
 {'page': 'page_19',
  'status': 'success',
  'output_file': '/home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_19/translated_en.md'},
 {'page': 'page_20',
  'status': 'success',
  'output_file': '/home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_20/translated_en.md'}]